In [3]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import unicodedata
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from pathlib import Path
import numpy as np

Mounted at /content/drive


In [5]:
from huggingface_hub import login
login()

In [6]:
#Some helper functions
from transformers import AutoModelForCausalLM

def load_causal_model(model_name):
    """Load an autoregressive/causal language model"""
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,  # Use half precision to save memory
        device_map='auto'  # use GPU
    )
    model.eval()
    print("Model loaded!")
    return tokenizer, model

In [7]:
class CausalLMScorer:

    def __init__(self, tokenizer, model):
        """Initialize the scorer"""
        self.tokenizer = tokenizer
        self.model = model

    def score_sentence(self, sentence):
        """Score a sentence using log-probability"""
        sentence = sentence.lower()
        input_ids = self.tokenizer.encode(sentence, return_tensors='pt')
        input_ids = input_ids.to(self.model.device)

        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            # outputs.loss is already the avg negative log-likelihood per token!
            avg_log_prob = -outputs.loss.item()

        num_tokens = input_ids.shape[1]
        avg_surprisal = -avg_log_prob

        return {
            'avg_log_prob': avg_log_prob,  #length-normalized
            'avg_surprisal': avg_surprisal,
            'num_tokens': num_tokens
        }

In [8]:
model_name = "openai-community/gpt2"
model_tokenizer, model = load_causal_model(model_name)
model_scorer = CausalLMScorer(model_tokenizer, model)

# Quick sanity test
gram = "Το παιδί τρώει"
ungram = "Το παιδιά τρώει"

result_gram = model_scorer.score_sentence(gram)
result_ungram = model_scorer.score_sentence(ungram)

print(f"Grammatical PLL: {result_gram['avg_log_prob']:.2f}")
print(f"Ungrammatical PLL: {result_ungram['avg_log_prob']:.2f}")
print(f"Model prefers grammatical: {result_gram['avg_log_prob'] > result_ungram['avg_log_prob']}")

Loading openai-community/gpt2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Grammatical PLL: -3.01
Ungrammatical PLL: -2.94
Model prefers grammatical: False


In [9]:
def load_phenomenon_pairs(phenomenon_path):
    """Load pairs for one phenomenon"""
    gram_file = phenomenon_path / "correct.txt"
    ungram_file = phenomenon_path / "incorrect.txt"

    with open(gram_file, 'r', encoding='utf-8') as f:
        gram_sentences = f.readlines()

    with open(ungram_file, 'r', encoding='utf-8') as f:
        ungram_sentences = f.readlines()

    pairs = list(zip(gram_sentences, ungram_sentences))
    return pairs

def load_all_phenomena(data_dir):
    """Load all phenomena"""
    all_pairs = {}
    data_path = Path(data_dir)

    for phenomenon_folder in data_path.iterdir():
        if phenomenon_folder.is_dir():
            phenomenon_name = phenomenon_folder.name

            try:
                pairs = load_phenomenon_pairs(phenomenon_folder)
                all_pairs[phenomenon_name] = pairs
                print(f"✓ Loaded {phenomenon_name}: {len(pairs)} pairs")
            except FileNotFoundError:
                print(f"✗ Skipped {phenomenon_name}: files not found")

    return all_pairs

In [10]:
# test loading all phenomena
all_data = load_all_phenomena("/content/drive/MyDrive/Thesis/data/phenomena")

print(f"\nTotal phenomena loaded: {len(all_data)}")
print("\nSummary:")
for phen_name, pairs in all_data.items():
    print(f"  - {phen_name}: {len(pairs)} pairs")

✓ Loaded noun_adjective_agreement: 100 pairs
✓ Loaded aspect: 100 pairs
✓ Loaded einai_agreement: 100 pairs
✓ Loaded subject_verb_agreement: 100 pairs
✓ Loaded negations: 100 pairs
✓ Loaded case_selection: 100 pairs

Total phenomena loaded: 6

Summary:
  - noun_adjective_agreement: 100 pairs
  - aspect: 100 pairs
  - einai_agreement: 100 pairs
  - subject_verb_agreement: 100 pairs
  - negations: 100 pairs
  - case_selection: 100 pairs


In [11]:
import random

def sample_size_stability(scorer, all_data, sample_sizes=[25, 50, 75, 100], n_repeats=5):
    results = {}
    for phenomenon, pairs in all_data.items():
        print(f"\nPhenomenon: {phenomenon} ({len(pairs)} pairs available)")
        pair_correctness = []
        for gram, ungram in pairs:
            g = scorer.score_sentence(gram)
            u = scorer.score_sentence(ungram)
            pair_correctness.append(g['avg_log_prob'] > u['avg_log_prob'])

        phen_results = {}
        for size in sample_sizes:
            if size > len(pair_correctness):
                continue
            accs = []
            for rep in range(n_repeats):
                random.seed(rep)
                sample = random.sample(pair_correctness, size)
                accs.append(sum(sample) / size)
            phen_results[size] = {
                'mean_accuracy': sum(accs) / len(accs),
                'min_accuracy': min(accs),
                'max_accuracy': max(accs),
                'all_runs': accs
            }
            print(f"  n={size}: mean={phen_results[size]['mean_accuracy']:.2%}")
        results[phenomenon] = phen_results
    return results

stability_results = sample_size_stability(model_scorer, all_data)


Phenomenon: noun_adjective_agreement (100 pairs available)
  n=25: mean=76.00%
  n=50: mean=72.40%
  n=75: mean=71.20%
  n=100: mean=70.00%

Phenomenon: aspect (100 pairs available)
  n=25: mean=61.60%
  n=50: mean=62.40%
  n=75: mean=64.53%
  n=100: mean=62.00%

Phenomenon: einai_agreement (100 pairs available)
  n=25: mean=60.00%
  n=50: mean=58.40%
  n=75: mean=56.80%
  n=100: mean=58.00%

Phenomenon: subject_verb_agreement (100 pairs available)
  n=25: mean=54.40%
  n=50: mean=54.80%
  n=75: mean=57.60%
  n=100: mean=59.00%

Phenomenon: negations (100 pairs available)
  n=25: mean=15.20%
  n=50: mean=18.00%
  n=75: mean=17.60%
  n=100: mean=17.00%

Phenomenon: case_selection (100 pairs available)
  n=25: mean=79.20%
  n=50: mean=78.40%
  n=75: mean=81.60%
  n=100: mean=82.00%


In [12]:
# Evaluate with ALL metrics tracked
all_results = {}

for phenomenon_name, pairs in all_data.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {phenomenon_name}")
    print(f"{'='*50}")

    correct_count = 0
    total_count = len(pairs)
    pair_results = []

    for gram, ungram in pairs:
        gram_result = model_scorer.score_sentence(gram)
        ungram_result = model_scorer.score_sentence(ungram)

        is_correct = gram_result['avg_log_prob'] > ungram_result['avg_log_prob']
        if is_correct:
            correct_count += 1

        pair_results.append({
            'grammatical': gram.strip(),
            'ungrammatical': ungram.strip(),
            'gram_avg_log_prob': gram_result['avg_log_prob'],
            'ungram_avg_log_prob': ungram_result['avg_log_prob'],
            'gram_surprisal': gram_result['avg_surprisal'],
            'ungram_surprisal': ungram_result['avg_surprisal'],
            'gram_num_tokens': gram_result['num_tokens'],
            'ungram_num_tokens': ungram_result['num_tokens'],
            'correct': is_correct
        })

    accuracy = correct_count / total_count
    avg_gram_log_prob = sum(p['gram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_ungram_log_prob = sum(p['ungram_avg_log_prob'] for p in pair_results) / len(pair_results)
    avg_gram_surprisal = sum(p['gram_surprisal'] for p in pair_results) / len(pair_results)
    avg_ungram_surprisal = sum(p['ungram_surprisal'] for p in pair_results) / len(pair_results)

    all_results[phenomenon_name] = {
        'correct': correct_count,
        'total': total_count,
        'accuracy': accuracy,
        'avg_gram_log_prob': avg_gram_log_prob,
        'avg_ungram_log_prob': avg_ungram_log_prob,
        'avg_gram_surprisal': avg_gram_surprisal,
        'avg_ungram_surprisal': avg_ungram_surprisal,
        'pairs': pair_results
    }

    print(f"Accuracy: {correct_count}/{total_count} = {accuracy:.2%}")
    print(f"Avg grammatical log prob: {avg_gram_log_prob:.2f}")
    print(f"Avg ungrammatical log prob: {avg_ungram_log_prob:.2f}")
    print(f"Avg grammatical surprisal: {avg_gram_surprisal:.2f}")
    print(f"Avg ungrammatical surprisal: {avg_ungram_surprisal:.2f}")

# Full summary table
print(f"\n{'='*90}")
print(f"COMPLETE SUMMARY")
print(f"{'='*90}")
print(f"{'Phenomenon':<25} {'Accuracy':<12} {'Gram LogP':<12} {'Ungram LogP':<12} {'Gram Surp':<12}")
print(f"{'-'*90}")
for phen, result in all_results.items():
    print(f"{phen:<25} {result['accuracy']:>10.2%} "
          f"{result['avg_gram_log_prob']:>11.2f} "
          f"{result['avg_ungram_log_prob']:>11.2f} "
          f"{result['avg_gram_surprisal']:>11.2f}")

total_correct = sum(r['correct'] for r in all_results.values())
total_pairs = sum(r['total'] for r in all_results.values())
overall_acc = total_correct / total_pairs
print(f"{'-'*90}")
print(f"{'Overall':<25} {overall_acc:>10.2%} ({total_correct}/{total_pairs})")


Evaluating: noun_adjective_agreement
Accuracy: 70/100 = 70.00%
Avg grammatical log prob: -3.06
Avg ungrammatical log prob: -3.08
Avg grammatical surprisal: 3.06
Avg ungrammatical surprisal: 3.08

Evaluating: aspect
Accuracy: 62/100 = 62.00%
Avg grammatical log prob: -2.46
Avg ungrammatical log prob: -2.48
Avg grammatical surprisal: 2.46
Avg ungrammatical surprisal: 2.48

Evaluating: einai_agreement
Accuracy: 58/100 = 58.00%
Avg grammatical log prob: -2.96
Avg ungrammatical log prob: -2.98
Avg grammatical surprisal: 2.96
Avg ungrammatical surprisal: 2.98

Evaluating: subject_verb_agreement
Accuracy: 59/100 = 59.00%
Avg grammatical log prob: -2.41
Avg ungrammatical log prob: -2.42
Avg grammatical surprisal: 2.41
Avg ungrammatical surprisal: 2.42

Evaluating: negations
Accuracy: 17/100 = 17.00%
Avg grammatical log prob: -3.22
Avg ungrammatical log prob: -3.07
Avg grammatical surprisal: 3.22
Avg ungrammatical surprisal: 3.07

Evaluating: case_selection
Accuracy: 82/100 = 82.00%
Avg gramma

In [13]:
import json
from datetime import datetime

output_dir = Path("/content/drive/MyDrive/Thesis/results/autoregressive")
output_dir.mkdir(parents=True, exist_ok=True)

results_to_save = {
    'model': model_name,
    'model_type': 'causal_lm',
    'timestamp': datetime.now().isoformat(),
    'overall': {'total_pairs': total_pairs, 'total_correct': total_correct, 'accuracy': overall_acc},
    'per_phenomenon': {
        phen: {
            'num_pairs': r['total'], 'correct': r['correct'], 'accuracy': r['accuracy'],
            'avg_grammatical_log_prob': r['avg_gram_log_prob'],
            'avg_ungrammatical_log_prob': r['avg_ungram_log_prob'],
            'avg_grammatical_surprisal': r['avg_gram_surprisal'],
            'avg_ungrammatical_surprisal': r['avg_ungram_surprisal'],
            'pairs': r['pairs']
        }
        for phen, r in all_results.items()
    }
}

results_file = output_dir / f"{model_name.replace('/', '_')}_results.json"
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results_to_save, f, ensure_ascii=False, indent=2)
print(f"✓ Saved: {results_file}")

stability_file = output_dir / f"{model_name.replace('/', '_')}_stability.json"
with open(stability_file, 'w', encoding='utf-8') as f:
    json.dump(stability_results, f, ensure_ascii=False, indent=2)
print(f"✓ Saved: {stability_file}")

✓ Saved: /content/drive/MyDrive/Thesis/results/autoregressive/openai-community_gpt2_results.json
✓ Saved: /content/drive/MyDrive/Thesis/results/autoregressive/openai-community_gpt2_stability.json
